In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io as sio

DATASET_ROOT = Path(r"G:\.shortcut-targets-by-id\1j00Dj_bVlyMXzgGql4TrxSMJxx-2WKtG\stage IAVC\Acquisition des données\Dataset")
ALL_CSV = Path(r".\BatchOutputs\ALL_hemiparetique_by_patient_rdv.csv")

PATIENT = "01-P-AR"   
RDV = "RDV2"
CONDITION = "Bare_pref"  # Bare_fast / Bare_pref / Shoe_fast / Shoe_pref

OUTDIR = Path("./QC_Outputs")
OUTDIR.mkdir(exist_ok=True)

from src.gait_processing import process_gait_data
from src.data_loader import apply_sensor_reorder, load_trial_from_mat

In [13]:
df = pd.read_csv(ALL_CSV)

# Missing values
miss = df.isna().sum().sort_values(ascending=False)
miss_pct = (miss / len(df) * 100).round(2)

print("QC SUMMARY")
print("==========")
print("Rows:", len(df), "| Cols:", df.shape[1])

qc_missing = pd.DataFrame({"missing": miss, "missing_%": miss_pct}).head(20)
display(qc_missing)

# Outliers (IQR) sur colonnes numériques
num_cols = df.select_dtypes(include=[np.number]).columns
outliers = {}
for c in num_cols:
    x = df[c].dropna().values
    if x.size < 8:
        continue
    q1, q3 = np.percentile(x, [25, 75])
    iqr = q3 - q1
    if iqr <= 0:
        continue
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers[c] = int(((df[c] < lo) | (df[c] > hi)).sum())

qc_out = pd.Series(outliers).sort_values(ascending=False).head(20).to_frame("outlier_count")
display(qc_out)

# Save txt (pratique pour rendre)
qc_txt = OUTDIR / "qc_summary.txt"
with open(qc_txt, "w", encoding="utf-8") as f:
    f.write("QC SUMMARY\n==========\n")
    f.write(f"Rows: {len(df)} | Cols: {df.shape[1]}\n\n")
    f.write("Missing values (top 20):\n")
    for c in qc_missing.index:
        f.write(f"- {c}: {int(miss[c])} ({miss_pct[c]}%)\n")
    f.write("\nOutliers (IQR) top 20:\n")
    for c, k in qc_out["outlier_count"].items():
        f.write(f"- {c}: {int(k)}\n")

print("Saved:", qc_txt)

QC SUMMARY
Rows: 16940 | Cols: 7


,missing,missing_%
Std,552,3.26
Patient,0,0.00
RDV,0,0.00
File,0,0.00
Parameter,0,0.00
Mean,0,0.00
Unit,0,0.00


,outlier_count
Std,1198
Mean,691


Saved: QC_Outputs\qc_summary.txt


In [15]:
TRIAL_NAMES = [
    "Bare_calibration", "Bare_fast", "Bare_pref",
    "Shoe_calibration", "Shoe_fast", "Shoe_pref"
]
NEW_ORDER = [7, 5, 6, 4, 3, 2, 1]

CONDITIONS = {
    "Bare_fast": (0, 1),
    "Bare_pref": (0, 2),
    "Shoe_fast": (3, 4),
    "Shoe_pref": (3, 5),
}

if CONDITION not in CONDITIONS:
    raise ValueError(f"CONDITION inconnue: {CONDITION}. Choix: {list(CONDITIONS.keys())}")

rdv_dir = DATASET_ROOT / "hemiparetique" / PATIENT / RDV
mat_path = rdv_dir / f"{PATIENT}_{RDV}_outputData.mat"
if not mat_path.exists():
    raise FileNotFoundError(f"Introuvable: {mat_path}")

print("Reading:", mat_path)
mat = sio.loadmat(str(mat_path), simplify_cells=True)
raw = mat["output"]

# trials
trials = []
for i in range(6):
    tr = load_trial_from_mat(raw[i])
    tr = apply_sensor_reorder(tr, NEW_ORDER)
    trials.append(tr)

# indices
indices = []
for name in TRIAL_NAMES:
    p = rdv_dir / f"{name}_indexData.mat"
    if not p.exists():
        raise FileNotFoundError(f"Introuvable: {p}")
    indices.append(sio.loadmat(str(p), simplify_cells=True))

calib_i, walk_i = CONDITIONS[CONDITION]
result = process_gait_data(trials[calib_i], indices[calib_i], trials[walk_i], indices[walk_i])

print("OK pipeline:", PATIENT, RDV, CONDITION)

Reading: G:\.shortcut-targets-by-id\1j00Dj_bVlyMXzgGql4TrxSMJxx-2WKtG\stage IAVC\Acquisition des données\Dataset\hemiparetique\01-P-AR\RDV2\01-P-AR_RDV2_outputData.mat
OK pipeline: 01-P-AR RDV2 Bare_pref


In [16]:
dbg = result["Debug"]
t = np.asarray(dbg["time"])

s1 = dbg["sensor1"]
s2 = dbg["sensor2"]

fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
fig.suptitle(f"{PATIENT} {RDV} - {CONDITION} | Extracted signals", fontsize=14)

def plot_sensor(ax_df, ax_h, sensor, name):
    Df  = np.asarray(sensor["Df"])
    Dfd = np.asarray(sensor["Df_dot"])
    P   = np.asarray(sensor["P"])
    Hto = np.asarray(sensor["Hto"])
    Hic = np.asarray(sensor["Hic"])

    ax_df.plot(t, Df, label="Df")
    ax_df.plot(t, Dfd, label="Df_dot")
    ax_df.plot(t, P, label="P (swing)")
    ax_df.set_title(name + ": Df / Df_dot / P")
    ax_df.grid(True, alpha=0.3)
    ax_df.legend()

    ax_h.plot(t, Hto, label="Hto")
    ax_h.plot(t, Hic, label="Hic")
    ax_h.set_title(name + ": Hto / Hic")
    ax_h.set_ylim(-0.1, 1.1)
    ax_h.grid(True, alpha=0.3)
    ax_h.legend()

plot_sensor(axes[0], axes[1], s1, "Sensor 1")
plot_sensor(axes[2], axes[3], s2, "Sensor 2")
axes[-1].set_xlabel("Time (ms)")

plt.tight_layout(rect=[0,0,1,0.95])
plt.show()

out_png = OUTDIR / f"Example_{PATIENT}_{RDV}_{CONDITION}_SwingSignals.png"
fig.savefig(out_png, dpi=180)
print("Saved:", out_png)

C:\Users\pc\AppData\Local\Temp\ipykernel_21988\2187403054.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved: QC_Outputs\Example_01-P-AR_RDV2_Bare_pref_SwingSignals.png


In [18]:
ang = dbg["angles"]
sw  = dbg["swing"]

ah = np.asarray(ang["ah"])      # hip affected
ak = np.asarray(ang["ak"])      # knee affected
nh = np.asarray(ang.get("nh", np.array([])))  # hip non-affected (si dispo)
nk = np.asarray(ang.get("nk", np.array([])))  # knee non-affected (si dispo)

P_af = np.asarray(sw["af"])     # swing mask (affected)

swing_mask = P_af > 0.5
stance_mask = ~swing_mask

# fallback si masque vide
if swing_mask.sum() < 5:
    swing_mask[:] = True
if stance_mask.sum() < 5:
    stance_mask[:] = True

# FG/FH/EH
fg_idx = np.nanargmax(np.where(swing_mask, ak, np.nan))   # knee max during swing
fh_idx = np.nanargmax(np.where(swing_mask, ah, np.nan))   # hip max during swing
eh_idx = np.nanargmin(np.where(stance_mask, ah, np.nan))  # hip min during stance

FG, FH, EH = float(ak[fg_idx]), float(ah[fh_idx]), float(ah[eh_idx])
print(f"FG={FG:.3f} | FH={FH:.3f} | EH={EH:.3f}")

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
fig.suptitle(f"{PATIENT} {RDV} - {CONDITION} | FG / FH / EH (marked)", fontsize=14)

axes[0].plot(t, ah, label="Hip (affected)")
if nh.size == t.size:
    axes[0].plot(t, nh, label="Hip (non-affected)", linewidth=1.0)

axes[0].scatter([t[fh_idx]], [FH])
axes[0].annotate(f"FH={FH:.2f}", (t[fh_idx], FH), textcoords="offset points", xytext=(10,10))
axes[0].scatter([t[eh_idx]], [EH])
axes[0].annotate(f"EH={EH:.2f}", (t[eh_idx], EH), textcoords="offset points", xytext=(10,-15))

axes[0].grid(True, alpha=0.3)
axes[0].legend()
axes[0].set_ylabel("Hip angle (deg)")

axes[1].plot(t, ak, label="Knee (affected)")
if nk.size == t.size:
    axes[1].plot(t, nk, label="Knee (non-affected)", linewidth=1.0)

axes[1].scatter([t[fg_idx]], [FG])
axes[1].annotate(f"FG={FG:.2f}", (t[fg_idx], FG), textcoords="offset points", xytext=(10,10))

# swing mask pour visualiser
axes[1].plot(t, 0.1*P_af + np.nanmin(ak), label="Swing (scaled)")

axes[1].grid(True, alpha=0.3)
axes[1].legend()
axes[1].set_ylabel("Knee angle (deg)")
axes[1].set_xlabel("")

plt.tight_layout(rect=[0,0,1,0.95])
plt.show()

out_png = OUTDIR / f"Example_{PATIENT}_{RDV}_{CONDITION}_Angles_FG_FH_EH.png"
fig.savefig(out_png, dpi=180)
print("Saved:", out_png)

# Résumé txt
ex_txt = OUTDIR / "example_summary.txt"
with open(ex_txt, "w", encoding="utf-8") as f:
    f.write(f"Patient: {PATIENT}\nRDV: {RDV}\nCondition: {CONDITION}\n\n")
    f.write(f"FG (knee max swing): {FG:.3f} deg\n")
    f.write(f"FH (hip  max swing): {FH:.3f} deg\n")
    f.write(f"EH (hip  min stance): {EH:.3f} deg\n")
print("Saved:", ex_txt)

FG=64.096 | FH=35.711 | EH=-22.548
Saved: QC_Outputs\Example_01-P-AR_RDV2_Bare_pref_Angles_FG_FH_EH.png
Saved: QC_Outputs\example_summary.txt


C:\Users\pc\AppData\Local\Temp\ipykernel_21988\4270255110.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
def corr(a, b):
    a = np.asarray(a); b = np.asarray(b)
    if a.size == 0 or b.size == 0 or a.size != b.size:
        return np.nan
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() < 5:
        return np.nan
    if np.nanstd(a[mask]) < 1e-6 or np.nanstd(b[mask]) < 1e-6:
        return np.nan
    return float(np.corrcoef(a[mask], b[mask])[0, 1])

aa = np.asarray(ang.get("aa", np.array([])))  # ankle affected
na = np.asarray(ang.get("na", np.array([])))  # ankle non-affected

hip_r = corr(ah, nh)
knee_r = corr(ak, nk)
ankle_r = corr(aa, na)

print("Left/Right consistency (affected vs non-affected):")
print(f"Hip  : {hip_r:.3f}")
print(f"Knee : {knee_r:.3f}")
print(f"Ankle: {ankle_r:.3f}")

Left/Right consistency (affected vs non-affected):
Hip  : 0.816
Knee : 0.683
Ankle: 0.027
